# Week 11, Day 4: Power spectra: finding rhythms in noisy signals

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w011/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2020)
fs = 250

def aperiodic(n, chi, rng):
    """Noise with power proportional to 1/f^chi, made by shaping white noise in the frequency domain."""
    X = np.fft.rfft(rng.normal(size=n)); f = np.fft.rfftfreq(n, 1 / fs)
    scale = np.zeros_like(f); scale[1:] = f[1:] ** (-chi / 2)
    y = np.fft.irfft(X * scale, n)
    return y / y.std()

def alpha_bursts(n, amp, rng):
    """10 Hz bursts with random timing, as alpha appears in real EEG."""
    t = np.arange(n) / fs
    env = np.convolve((rng.random(n) < 0.004).astype(float), np.hanning(int(0.8 * fs)), "same")
    return amp * np.clip(env, 0, 1) * np.sin(2 * np.pi * 10 * t + rng.uniform(0, 2 * np.pi))

N = 60 * fs                                            # one minute
eyes_closed = aperiodic(N, 1.5, rng) + alpha_bursts(N, 3.0, rng)
eyes_open = aperiodic(N, 1.5, rng) + alpha_bursts(N, 0.5, rng)
plt.figure(figsize=(7, 2.5)); plt.plot(np.arange(2 * fs) / fs, eyes_closed[:2 * fs], lw=0.8, label="eyes closed"); plt.plot(np.arange(2 * fs) / fs, eyes_open[:2 * fs] - 6, lw=0.8, label="eyes open (offset)")
plt.xlabel("time (s)"); plt.legend(frameon=False); plt.show()

### Exercise: A periodogram in physical units

Write `periodogram(x, fs)` returning frequencies and the PSD estimate: remove the mean, take `rfft`, compute $|X_k|^2/(f_sN)$ and double every bin except the first and last. Check that it integrates to the variance, and plot the eyes-closed spectrum on log-log axes.

In [ ]:
def periodogram(x, fs):
    x = x - x.mean()
    X = np.fft.rfft(x)
    p = ...
    p[1:-1] *= 2
    return np.fft.rfftfreq(len(x), 1 / fs), p

f, p = periodogram(eyes_closed, fs)
plt.figure(figsize=(5, 3.2)); plt.loglog(f[1:], p[1:], lw=0.5); plt.xlabel("frequency (Hz)"); plt.ylabel("PSD (units² / Hz)"); plt.show()
print("integral of PSD:", round(np.sum(p) * (f[1] - f[0]), 4), "  variance:", round(np.var(eyes_closed), 4))

In [ ]:
import numpy as np
def test_units():
    "the PSD integrates to the variance of the signal (Parseval)"
    f, p = periodogram(eyes_closed, fs)
    assert np.isclose(np.sum(p) * (f[1] - f[0]), np.var(eyes_closed), rtol=1e-6)

def test_sine():
    "a unit sinusoid's power, summed over its peak, is 0.5"
    t = np.arange(4 * fs) / fs
    f, p = periodogram(np.sin(2 * np.pi * 20 * t), fs)
    assert np.isclose(np.sum(p) * (f[1] - f[0]), 0.5, rtol=1e-6)

test_units()
test_sine()
print("All checks passed")

### Exercise: Welch's method

Write `welch(x, fs, nperseg)`: Hann-windowed segments of length `nperseg`, half-overlapping, each with its mean removed, PSD $2|X_k|^2/(f_s\sum w^2)$ (not doubled at the first and last bins), averaged. Measure how jagged each estimate is by the variance of the differences of log10 PSD between neighbouring bins from 40 to 100 Hz (`jag_periodogram`, `jag_welch`).

In [ ]:
def welch(x, fs, nperseg):
    w = np.hanning(nperseg)
    step = nperseg // 2
    psds = []
    for start in range(0, len(x) - nperseg + 1, step):
        seg = x[start:start + nperseg]
        seg = seg - seg.mean()
        X = np.fft.rfft(w * seg)
        p = ...
        p[1:-1] *= 2
        psds.append(p)
    return np.fft.rfftfreq(nperseg, 1 / fs), np.mean(psds, axis=0)

fw, pw = welch(eyes_closed, fs, 512)
def jaggedness(f, p):
    m = (f > 40) & (f < 100)
    return np.var(np.diff(np.log10(p[m])))
jag_periodogram = jaggedness(f, p)
jag_welch = jaggedness(fw, pw)

plt.figure(figsize=(5, 3.2)); plt.loglog(f[1:], p[1:], lw=0.4, alpha=0.5, label="periodogram"); plt.loglog(fw[1:], pw[1:], lw=2, label="Welch, 512-sample segments")
plt.xlabel("frequency (Hz)"); plt.ylabel("PSD"); plt.legend(frameon=False); plt.show()
print(f"bin-to-bin jaggedness: periodogram {jag_periodogram:.3f}, Welch {jag_welch:.4f}")

In [ ]:
import numpy as np
def test_sine_power():
    "welch preserves the power of a sinusoid at 20 Hz (within 5%)"
    t = np.arange(20 * fs) / fs
    f, p = welch(np.sin(2 * np.pi * 20 * t), fs, 500)
    assert abs(np.sum(p) * (f[1] - f[0]) - 0.5) < 0.025

def test_resolution():
    "a segment of 512 samples gives a frequency spacing of fs / 512"
    f, p = welch(eyes_closed, fs, 512)
    assert np.isclose(f[1] - f[0], fs / 512)

def test_smoother():
    "Welch's estimate fluctuates far less from bin to bin than the periodogram's"
    assert jag_welch < jag_periodogram / 20

test_sine_power()
test_resolution()
test_smoother()
print("All checks passed")

### Exercise: Fit the background, then measure alpha

Write `fit_aperiodic(f, p)`, a straight-line fit of log10 PSD against log10 frequency over 2 to 40 Hz, excluding the alpha range 7 to 14 Hz. For both conditions (Welch, 512-sample segments), store the exponent (minus the slope) in `chi_closed` and `chi_open`, and the mean height of log10 PSD above the fitted line from 8 to 12 Hz in `alpha_closed` and `alpha_open`.

In [ ]:
def fit_aperiodic(f, p):
    sel = (f >= 2) & (f <= 40) & ~((f >= 7) & (f <= 14))
    slope, icpt = ...
    return slope, icpt

def analyse(x):
    f, p = welch(x, fs, 512)
    slope, icpt = fit_aperiodic(f, p)
    a = (f >= 8) & (f <= 12)
    excess = ...
    return -slope, excess, f, p, slope, icpt

chi_closed, alpha_closed, fc, pc, sc, ic = analyse(eyes_closed)
chi_open, alpha_open, fo, po, so, io = analyse(eyes_open)

plt.figure(figsize=(5, 3.2))
for f_, p_, s_, i_, lab in [(fc, pc, sc, ic, "eyes closed"), (fo, po, so, io, "eyes open")]:
    line, = plt.loglog(f_[1:], p_[1:], label=lab)
    plt.loglog(f_[1:], 10 ** (i_ + s_ * np.log10(f_[1:])), "--", color=line.get_color(), lw=1)
plt.xlim(1, 100); plt.xlabel("frequency (Hz)"); plt.ylabel("PSD"); plt.legend(frameon=False); plt.show()
print(f"exponents: closed {chi_closed:.2f}, open {chi_open:.2f};  alpha above background (log10 units): closed {alpha_closed:.2f}, open {alpha_open:.2f}")

In [ ]:
import numpy as np
def test_fit():
    "fit_aperiodic returns the slope and intercept of log10 PSD against log10 frequency, excluding 7 to 14 Hz"
    slope, icpt = fit_aperiodic(fw, pw)
    sel = (fw >= 2) & (fw <= 40) & ~((fw >= 7) & (fw <= 14))
    s2, i2 = np.polyfit(np.log10(fw[sel]), np.log10(pw[sel]), 1)
    assert np.isclose(slope, s2) and np.isclose(icpt, i2)

def test_exponent():
    "the estimated exponents are within 0.25 of the true 1.5 in both conditions"
    assert abs(chi_closed - 1.5) < 0.25 and abs(chi_open - 1.5) < 0.25

def test_alpha():
    "alpha rises far more above the background with eyes closed"
    assert alpha_closed > alpha_open + 0.8

test_fit()
test_exponent()
test_alpha()
print("All checks passed")